### Preparo de dados

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# diretório dados
base_path = Path('./Dados_EEG/eegmmidb')

In [3]:
def tira_janelas(X, y, sub_processar, run_processar, base_path, canais=None):
    if not base_path.is_dir():
        print(f"Erro: O diretório '{base_path}' não foi encontrado.")
        print(f"Diretório atual: '{os.getcwd()}'")
        return
    
    else:
        print(f"Iniciando processamento dos dados de: {base_path.resolve()}\n")

    # --- LOOP PRINCIPAL ---
    for subject_id in sub_processar:
        for run_id in run_processar:
            
            s_id_str = f"{subject_id:03d}" # ex: 001
            r_id_str = f"{run_id:02d}"     # ex: 01

            ann_filename = f"SUB_{s_id_str}_ANN_{r_id_str}.csv"
            sig_filename = f"SUB_{s_id_str}_SIG_{r_id_str}.csv"

            ann_filepath = base_path / ann_filename
            sig_filepath = base_path / sig_filename

                
            if ann_filepath.exists() and sig_filepath.exists():
                #print(f"Processando: Sujeito {s_id_str}, Run {r_id_str}")

                try:
                    
                    colunas_ann = ['rotulo', 'duracao_s', 'tamanho_amostras', 'amostra_inicio', 'amostra_final']
                    df_ann = pd.read_csv(ann_filepath, header=None, names=colunas_ann)
                    df_sig = pd.read_csv(sig_filepath, header=None)

                    # --- EXTRACT EGMENTOS E RÓTULOS ---
                    for index, row in df_ann.iterrows():
                        
                        rotulo = row['rotulo']
                        inicio = row['amostra_inicio']
                        fim = row['amostra_final']

                        inicio = int(inicio)-1 # idx começa zero
                        fim = int(fim)

                        # Seleciona os canais SE fornecidos
                        if canais:
                            segmento_sinal = df_sig.iloc[inicio:fim, canais].values
                        else: # manda tudo msm
                            segmento_sinal = df_sig.iloc[inicio:fim].values

                        X.append(segmento_sinal)

                        if (rotulo==1 or rotulo==4):
                            y.append(0)
                        elif (rotulo==2 or rotulo==5):
                            y.append(1)
                        elif (rotulo==3 or rotulo==6):
                            y.append(2)

                except Exception as e:
                    print(f"  -> Erro ao processar o par de arquivos para S{s_id_str}/R{r_id_str}: {e}")
            else:
                # arquiv não encontrado
                print(f"Aviso: Par de arquivos para Sujeito {s_id_str}, Run {r_id_str} não encontrado. Pulando.")
                pass

    print("\n--- Processamento Concluído ---")
        
    y = np.array(y) # convt array NumPy

    print(f"Total de segmentos extraídos: {len(X)}")
    print(f"Total de rótulos extraídos: {len(y)}")
    print(f"Exemplo de um segmento (shape): {X[0].shape if X else 'N/A'}")
    print(f"Rótulos únicos encontrados: {np.unique(y) if y.size > 0 else 'N/A'}")

[Imagem da disposição dos 64 Canais](64_channel_sharbrough.png).

In [4]:
X_train, X_test = [], [] #segmento sinal
y_train, y_test = [], [] #rótulos

sub_train = range(1, 81) # 1 a 80
sub_test = range(81,104) #81 a 103

run_move = [1, 5, 9]
run_imag = [2, 6, 10] 
run_total = run_move + run_imag  

CANAIS = [32, 9, 11, 13, 38] #  F3, C3, Cz, C4, F4
 
base_path = Path('./Dados_EEG/eegmmidb')
tira_janelas(X_train, y_train, sub_train, run_total, base_path, canais= CANAIS)
tira_janelas(X_test, y_test, sub_test, run_total, base_path, canais= CANAIS)

Iniciando processamento dos dados de: C:\Users\eduar\TCC\TCC-EEG_Control\Programa_CNN\Dados_EEG\eegmmidb


--- Processamento Concluído ---
Total de segmentos extraídos: 14400
Total de rótulos extraídos: 14400
Exemplo de um segmento (shape): (672, 5)
Rótulos únicos encontrados: [0 1 2]
Iniciando processamento dos dados de: C:\Users\eduar\TCC\TCC-EEG_Control\Programa_CNN\Dados_EEG\eegmmidb


--- Processamento Concluído ---
Total de segmentos extraídos: 4140
Total de rótulos extraídos: 4140
Exemplo de um segmento (shape): (656, 5)
Rótulos únicos encontrados: [0 1 2]


### Seleção Dados para utilizar

In [5]:
def veri_tam_janelas (sinais):
    tamanhos_das_janelas = [janela.shape[0] for janela in sinais]

    print("\n--- Verificação dos Tamanhos das Janelas ---")
    print(f"Total de janelas extraídas: {len(tamanhos_das_janelas)}")
    
    print("--- Resumo do Tamanho dos Sinais ---")
    print(f"Tamanho Mínimo: {np.min(tamanhos_das_janelas)} amostras")
    print(f"Tamanho Máximo: {np.max(tamanhos_das_janelas)} amostras")


sinais = X_train + X_test
veri_tam_janelas(sinais)


--- Verificação dos Tamanhos das Janelas ---
Total de janelas extraídas: 18540
--- Resumo do Tamanho dos Sinais ---
Tamanho Mínimo: 416 amostras
Tamanho Máximo: 688 amostras


#### Ajusta tamanho das Janelas

In [6]:
# Corte das janelas Maiores
def recorte_central(segmento, tamanho_max):
    """
    Aplica um recorte central a um segmento para que ele se ajuste 
    ao tamanho_maximo.
    """
    compr_atual = segmento.shape[0]
    
    if compr_atual <= tamanho_max: # se menor ta ok
        return segmento 
        
    excesso = compr_atual - tamanho_max
    corte_inicio = excesso // 2 # força divisão inteira
    indice_final = corte_inicio + tamanho_max
    janela_cortada = segmento[corte_inicio:indice_final, :]
    
    return janela_cortada

In [7]:
#Preenchendo janelas Menores
def preenche_janela(segmento, tamanho_min):
    """
    Adiciona zeros a um segmento para que ele se ajuste 
    ao tamanho_minimo.
    """
    compr_atual = segmento.shape[0]
    
    if compr_atual >= tamanho_min: # se maior ta ok
        return segmento 

    num_canais = segmento.shape[1]     
    faltante = tamanho_min - compr_atual
    padding = np.zeros((faltante, num_canais)) # adiciona zeros

    janela_aumentada = np.vstack([segmento, padding])
    
    return janela_aumentada

Aplicando Ajuste

In [8]:
def ajustar_janelas(lista_sinais, tamanho_alvo):
    """
    Aplica o recorte central e o preenchimento em uma lista de sinais.
    """
    sinal_ajustado = []
    for segmento in lista_sinais:
        segmento_cortado = recorte_central(segmento, tamanho_alvo)
        segmento_preench = preenche_janela(segmento_cortado, tamanho_alvo) 
        sinal_ajustado.append(segmento_preench)
    return sinal_ajustado

In [9]:
TAM_JANELA = 640  # 160hz * 4seg

X_train = ajustar_janelas(X_train, TAM_JANELA)
X_test  = ajustar_janelas(X_test, TAM_JANELA)

veri_tam_janelas(X_train), veri_tam_janelas(X_test)


--- Verificação dos Tamanhos das Janelas ---
Total de janelas extraídas: 14400
--- Resumo do Tamanho dos Sinais ---
Tamanho Mínimo: 640 amostras
Tamanho Máximo: 640 amostras

--- Verificação dos Tamanhos das Janelas ---
Total de janelas extraídas: 4140
--- Resumo do Tamanho dos Sinais ---
Tamanho Mínimo: 640 amostras
Tamanho Máximo: 640 amostras


(None, None)

---
Tirando exemplos pra C

In [57]:
print(f"X_test Max:{np.max(X_test)}, Min:{np.min(X_test)}")

X_test Max:805.0, Min:-678.0


In [52]:
indice_desejado = 11
janela_especifica = X_test[indice_desejado]
janela_achatada = janela_especifica.flatten()
rotulo_correspondente = y_test[indice_desejado]


print(f"Shape da janela {indice_desejado}: {janela_especifica.shape}")
print(f"Shape da janela {indice_desejado}: {janela_achatada.shape}")
print(f"Dtype da janela achatada {janela_achatada.dtype}")
print(f"Max:{max(janela_achatada)}, Min:{min(janela_achatada)}")
print(f"Rótulo: {rotulo_correspondente}")

Shape da janela 11: (640, 5)
Shape da janela 11: (3200,)
Dtype da janela achatada int64
Max:128, Min:-115
Rótulo: 2


In [ ]:
# param Normal
MEDIA_TREINO = np.array([-6.503685, -3.132349, -1.297647, -4.060297, -2.540005], dtype=np.float32)
DESVIO_PADRAO_TREINO = np.array([73.443481, 59.480194, 57.704811, 59.828823, 67.497391], dtype=np.float32)

# param Quant
INPUT_SCALE = 3.9294116497039795
INPUT_ZERO_POINT = 9


def normalize_sample(sample_float_raw, mean, std_dev):
    """Aplica a normalização (subtrai média, divide por desvio padrão)."""
    # Garante float32 e aplica a normalização canal a canal
    return ((sample_float_raw.astype(np.float32) - mean) / std_dev)

def quantize_sample(sample_float_normalized, scale, zero_point):
    """Converte uma amostra float32 normalizada para int8 quantizado."""
    sample_quantized = (sample_float_normalized / scale) + zero_point
    sample_quantized = np.round(sample_quantized)
    sample_quantized = np.clip(sample_quantized, -128, 127) 
    return sample_quantized.astype(np.int8)

In [ ]:

    
print(f"--- Processando Amostra {indice_desejado} ---")
print(f"Dados Brutos (int64) - Max: {np.max(janela_especifica)}, Min: {np.min(janela_especifica)}")


janela_normalizada = normalize_sample(janela_especifica, MEDIA_TREINO, DESVIO_PADRAO_TREINO)

janela_quantizada = quantize_sample(janela_normalizada, INPUT_SCALE, INPUT_ZERO_POINT)
janela_achatada = janela_quantizada.flatten()

print(f"\nDados Processados (int8) - Max: {np.max(janela_achatada)}, Min: {np.min(janela_achatada)}")

--- Processando Amostra 11 ---
Dados Brutos (int64) - Max: 128, Min: -115

Dados Processados (int8) - Max: 10, Min: 9


In [56]:
nome_arquivo_c = 'janela_teste_array.h'
nome_array_c = f'teste_array_{rotulo_correspondente}' # Nome da variável no C

with open(nome_arquivo_c, 'w') as f:
    f.write(f"#ifndef JANELA_TESTE_ARRAY_H_\n")
    f.write(f"#define JANELA_TESTE_ARRAY_H_\n\n")
    f.write(f"#include <stdint.h> // Para usar int8_t\n\n")
    
    f.write(f"const int8_t {nome_array_c}[{len(janela_achatada)}] = {{\n")
    
    # Escreve os valores formatados, 12 por linha
    for i in range(0, len(janela_achatada), 12):
        line_values = janela_achatada[i:i+12]
        line_str = ", ".join(map(str, line_values))
        f.write(f"  {line_str},\n")
        
    f.write(f"}};\n\n")
    f.write(f"#endif // JANELA_TESTE_ARRAY_H_\n")

print(f"Array C salvo em '{nome_arquivo_c}' com o nome '{nome_array_c}'")

Array C salvo em 'janela_teste_array.h' com o nome 'teste_array_2'


Fim da geração pra C

---

#### Data Augmentation (tvz use)

In [ ]:
# Parâmetro de ruído 
noise_factor = 0.01

# Parâmetro de escala
scale_factor = 0.1  

# Deslocar o sinal no tempo
def shift_signal(data, maDADOS_X_shift):
    shift = np.random.randint(-maDADOS_X_shift, maDADOS_X_shift)
    return np.roll(data, shift, aDADOS_Xis=0)

shift_maDADOS_X = 20  # Máximo deslocamento

from scipy.ndimage import gaussian_filter1d

# jitter, escalonamento, deslocamento e suavização
X_noise = DADOS_X + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=DADOS_X.shape)
X_aug = X_noise * (1 + scale_factor * np.random.uniform(-1, 1, size=DADOS_X.shape))
X_aug = gaussian_filter1d(DADOS_X, sigma=2, aDADOS_Xis=1) 
X_aug = np.array([shift_signal(dado, shift_maDADOS_X) for dado in X_aug])

# Filtra rotulos a aumentar
# X_menos = DADOS_X[DADOS_Y != 0]
# Y_menos = DADOS_Y[DADOS_Y != 0]

# # data augmentation para equilibrar rotulos
# X_noise = X_menos + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_menos.shape)
# X_aug = X_noise * (1 + scale_factor * np.random.uniform(-1, 1, size=X_noise.shape))
# X_aug = gaussian_filter1d(X_aug, sigma=2, aDADOS_Xis=1)
# X_aug = np.array([shift_signal(dado, shift_maDADOS_X) for dado in X_aug])


X_aug = np.concatenate([DADOS_X, X_aug], aDADOS_Xis=0)
y_aug = np.concatenate([DADOS_Y, DADOS_Y], aDADOS_Xis=0)

print(f"Forma dos dados: {X_aug.shape}")
print(f"Forma dos rótulos: {y_aug.shape}")

# DADOS_X = X_aug
# DADOS_Y = y_aug

### Separação Dados Treino e Validação

In [ ]:
print(f"Tamanho Treino: {len(X_train)}")
print(f"Tamanho Teste: {len(X_test)}")
print(f"Proporção: {len(X_train) / (len(X_train)+len(X_test)):.2%}%")

Convertendo para array NumPy

In [ ]:
import numpy as np

X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"Dimensão X_test:\n{X_test.shape}\nQnt, Compr, Canais")

Normalização será efetuada diretamente numa camada da rede

In [ ]:
# # NORMALIZAÇÃO 
# from sklearn.preprocessing import StandardScaler

# print("Normalizando os dados ...")
# scaler = StandardScaler()

# # Remod 2D para o scaler
# X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
# X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])

# # Scaler 
# scaler.fit(X_train_reshaped) # APENAS nos dados treino

# # Transformar ambos 
# X_train_scaled_reshaped = scaler.transform(X_train_reshaped)
# X_test_scaled_reshaped = scaler.transform(X_test_reshaped)

# # Remod3D para a rede
# X_train_scaled = X_train_scaled_reshaped.reshape(X_train.shape)
# X_test_scaled = X_test_scaled_reshaped.reshape(X_test.shape)

# print("Dados normalizados com sucesso!")

In [ ]:
import pickle

with open('Dados_EEG/dados_EEG1.pkl', mode='wb') as f:
    pickle.dump([X_train, y_train, X_test, y_test, CANAIS, TAM_JANELA], f)

NameError: name 'X_train' is not defined